# Error analysis

This notebook inspects per-run predictions from `runs/final_run`: confidence on correct vs incorrect labels, and accuracy by HateCheck functionality.

Predictions are joined to `processed_hatecheck.csv` on `case_id` so each test case keeps its functionality tag. Evaluation labels come from the prediction files (including Reddit/Bulgaria relabeling).

**Inputs:**
- `data/datasets/HateCheck/processed_hatecheck.csv`
- `runs/final_run/{dataset}/{model}/{definition}/{prompting}/predictions.csv`

**Outputs (under `tables/`):**
- `experiments_confidence.csv` — mean confidence overall and on correct/incorrect cases
- `experiments_accuracy_per_functionality.csv` — per-run accuracy by functionality, plus a mean row


## Setup

Load the processed HateCheck metadata and every prediction file in the paper grid (3 datasets × 3 models × 2 definitions × 4 prompting strategies).


In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RUN_DIR = PROJECT_ROOT / "runs" / "final_run"
TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(exist_ok=True)

HATECHECK_CSV = PROJECT_ROOT / "data" / "datasets" / "HateCheck" / "processed_hatecheck.csv"
DATASETS = ["hatecheck", "hatecheck_bulgaria", "hatecheck_reddit"]
SCOPES = {
    "hatecheck": ["hatecheck_no_definition", "hatecheck_hsc"],
    "hatecheck_bulgaria": ["bulgaria_no_definition", "bulgaria_hsc"],
    "hatecheck_reddit": ["reddit_no_definition", "reddit_hsc"],
}
MODELS = ["gemma2_2b_it", "llama3_2_3b_instruct", "qwen2_5_3b_instruct"]
PROMPTINGS = [
    "zero-shot",
    "few-shot-random",
    "few-shot-diverse",
    "few-shot-nearest-query",
]

hatecheck_df = pd.read_csv(HATECHECK_CSV)
print(f"Loaded {len(hatecheck_df):,} HateCheck rows from {HATECHECK_CSV.name}")
hatecheck_df.head()


Loaded 3,634 HateCheck rows from processed_hatecheck.csv


,case_id,functionality,test_case,label_gold,target_ident,direction,focus_words,focus_lemma,ref_case_id,ref_templ_id,templ_id,case_templ,target_type,dominance,explicit_ref,incites,group_insult,in_group
0,1,derog_neg_emote_h,I hate women.,hateful,women,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,gender,no,group_characteristic,hate,no,NaN
1,2,derog_neg_emote_h,I hate trans people.,hateful,trans people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,gender,no,group_characteristic,hate,no,NaN
2,3,derog_neg_emote_h,I hate gay people.,hateful,gay people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,sexual orientation,no,group_characteristic,hate,no,NaN
3,4,derog_neg_emote_h,I hate black people.,hateful,black people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,race,no,group_characteristic,hate,no,NaN
4,5,derog_neg_emote_h,I hate disabled people.,hateful,disabled people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,disability,no,group_characteristic,hate,no,NaN


In [2]:
predictions = {}
missing = []
for dataset in DATASETS:
    for model in MODELS:
        for definition in SCOPES[dataset]:
            for prompting in PROMPTINGS:
                path = RUN_DIR / dataset / model / definition / prompting / "predictions.csv"
                key = (dataset, model, definition, prompting)
                if path.is_file():
                    predictions[key] = pd.read_csv(path, dtype={"id": int})
                else:
                    missing.append(path)

print(f"Loaded {len(predictions)} prediction files from {RUN_DIR}")
if missing:
    print(f"Skipped {len(missing)} missing files")


Loaded 72 prediction files from /Users/parham/Desktop/Research Project/project/runs/final_run


## Confidence by setting

Mean model confidence overall, on correctly labeled cases, and on incorrectly labeled cases.


In [3]:
def confidence_row(preds: pd.DataFrame) -> dict:
    correct = preds["label"] == preds["prediction"]
    return {
        "overall_confidence": preds["confidence_score"].mean(),
        "confidence_correctly_labeled": preds.loc[correct, "confidence_score"].mean(),
        "confidence_incorrectly_labeled": preds.loc[~correct, "confidence_score"].mean(),
    }


confidence_rows = [
    {
        "dataset": dataset,
        "model": model,
        "definition": definition,
        "prompting": prompting,
        **confidence_row(preds),
    }
    for (dataset, model, definition, prompting), preds in predictions.items()
]
experiments_confidence = pd.DataFrame(confidence_rows).set_index(
    ["dataset", "model", "definition", "prompting"]
)
experiments_confidence.to_csv(TABLES_DIR / "experiments_confidence.csv")
print(f"Wrote {TABLES_DIR / 'experiments_confidence.csv'}")
experiments_confidence.round(3)


Wrote tables/experiments_confidence.csv


overall_confidence  \
dataset          model               definition              prompting                                    
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                            0.996   
                                                             few-shot-random                      0.997   
                                                             few-shot-diverse                     0.995   
                                                             few-shot-nearest-query               0.996   
                                     hatecheck_hsc           zero-shot                            0.995   
...                                                                                                 ...   
hatecheck_reddit qwen2_5_3b_instruct reddit_no_definition    few-shot-nearest-query               0.973   
                                     reddit_hsc              zero-shot                            0.969   
                                                             few-shot-random                      0.970   
                                                             few-shot-diverse                     0.957   
                                                             few-shot-nearest-query               0.978   

                                                                                     confidence_correctly_labeled  \
dataset          model               definition              prompting                                              
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                                      0.998   
                                                             few-shot-random                                0.999   
                                                             few-shot-diverse                               0.996   
                                                             few-shot-nearest-query                         0.998   
                                     hatecheck_hsc           zero-shot                                      0.996   
...                                                                                                           ...   
hatecheck_reddit qwen2_5_3b_instruct reddit_no_definition    few-shot-nearest-query                         0.979   
                                     reddit_hsc              zero-shot                                      0.981   
                                                             few-shot-random                                0.978   
                                                             few-shot-diverse                               0.965   
                                                             few-shot-nearest-query                         0.984   

                                                                                     confidence_incorrectly_labeled  
dataset          model               definition              prompting                                               
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                                        0.990  
                                                             few-shot-random                                  0.991  
                                                             few-shot-diverse                                 0.984  
                                                             few-shot-nearest-query                           0.982  
                                     hatecheck_hsc           zero-shot                                        0.987  
...                                                                                                             ...  
hatecheck_reddit qwen2_5_3b_instruct reddit_no_definition    few-shot-nearest-query                           0.952  
                                     reddit_hsc              zero-shot                                       

## Accuracy by HateCheck functionality

For each run, join predictions to HateCheck functionality tags and compute accuracy. Columns are ordered by mean accuracy (lowest first). A `mean_accuracy` row averages over the loaded configurations.


In [4]:
def accuracy_by_functionality(preds: pd.DataFrame) -> pd.Series:
    merged = hatecheck_df[["case_id", "functionality"]].merge(
        preds, left_on="case_id", right_on="id", how="inner"
    )
    correct = merged["label"] == merged["prediction"]
    return correct.groupby(merged["functionality"]).mean().rename("accuracy")


accuracy_rows = []
for (dataset, model, definition, prompting), preds in predictions.items():
    acc = accuracy_by_functionality(preds)
    row = acc.to_dict()
    row.update(
        {
            "dataset": dataset,
            "model": model,
            "definition": definition,
            "prompting": prompting,
        }
    )
    accuracy_rows.append(row)

index_cols = ["dataset", "model", "definition", "prompting"]
accuracy_df = pd.DataFrame(accuracy_rows).set_index(index_cols).round(3)
accuracy_df = accuracy_df[accuracy_df.mean().sort_values().index]
accuracy_df.loc[("mean_accuracy", "", "", "")] = accuracy_df.mean().round(3)
accuracy_df.to_csv(TABLES_DIR / "experiments_accuracy_per_functionality.csv")
print(f"Wrote {TABLES_DIR / 'experiments_accuracy_per_functionality.csv'}")
accuracy_df


Wrote tables/experiments_accuracy_per_functionality.csv


target_indiv_nh  \
dataset          model               definition              prompting                                 
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                         0.015   
                                                             few-shot-random                   0.138   
                                                             few-shot-diverse                  0.323   
                                                             few-shot-nearest-query            0.077   
                                     hatecheck_hsc           zero-shot                         0.400   
...                                                                                              ...   
hatecheck_reddit qwen2_5_3b_instruct reddit_hsc              zero-shot                         0.477   
                                                             few-shot-random                   0.662   
                                                             few-shot-diverse                  0.815   
                                                             few-shot-nearest-query            0.738   
mean_accuracy                                                                                  0.396   

                                                                                     counter_quote_nh  \
dataset          model               definition              prompting                                  
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                          0.213   
                                                             few-shot-random                    0.053   
                                                             few-shot-diverse                   0.295   
                                                             few-shot-nearest-query             0.242   
                                     hatecheck_hsc           zero-shot                          0.372   
...                                                                                               ...   
hatecheck_reddit qwen2_5_3b_instruct reddit_hsc              zero-shot                          0.464   
                                                             few-shot-random                    0.700   
                                                             few-shot-diverse                   0.773   
                                                             few-shot-nearest-query             0.787   
mean_accuracy                                                                                   0.494   

                                                                                     target_group_nh  \
dataset          model               definition              prompting                                 
hatecheck        gemma2_2b_it        hatecheck_no_definition zero-shot                         0.016   
                                                             few-shot-random                   0.145   
                                                             few-shot-diverse                  0.355   
                                                             few-shot-nearest-query            0.081   
                                     hatecheck_hsc           zero-shot                         0.484   
...                                                                                              ...   
hatecheck_reddit qwen2_5_3b_instruct reddit_hsc              zero-shot                         0.629   
                                                             few-shot-random                   0.742   
                                                             few-shot-diverse                  0.855   
                                                             few-shot-nearest-query            0.548   
mean_accuracy                                                                                  0.503   

             

## Lowest-scoring functionalities

Mean accuracy over configurations for the five weakest functionalities (the view used in the paper).


In [11]:
accuracy_df.iloc[-1, :10] * 100

target_indiv_nh      39.6
counter_quote_nh     49.4
target_group_nh      50.3
counter_ref_nh       53.9
derog_impl_h         69.4
slur_reclaimed_nh    69.7
ref_subs_sent_h      70.5
ref_subs_clause_h    71.3
slur_h               71.5
phrase_question_h    72.2
Name: (mean_accuracy, , , ), dtype: float64